# Time Series Decomposition

In this tutorial, we explore time series decomposition using the `normet` package.
The goal of decomposition is to split an observed signal into interpretable 
components such as:

- **Weather component**: Decompose a time series into meteorological contributions ranked by model
    feature importance.
- **Emission component**: Decompose a time series into emission-related components by progressively
    freezing time-related variables during resampling.


# 1. Load Dataset and model

In [1]:
import normet as nm
from _synth import make_my1_data

features_to_use = [
    "u10", "v10", "d2m", "t2m", "blh", "sp", "ssrd", "tcc", "tp", "rh2m"
]
my1_raw = make_my1_data().set_index('date')
df_pre = nm.prepare_data(my1_raw, value='PM2.5', feature_names=features_to_use)
df_pre = df_pre.set_index('date')
model_flaml = nm.load_model(path='.', filename='automl.joblib')
model_lgb   = nm.load_model(path='.', backend='lightgbm', filename='automl_lgb.joblib')

In [2]:
# Shared feature configuration used throughout this notebook
feature_names = [
    "u10", "v10", "d2m", "t2m", "blh", "sp", "ssrd", "tcc", "tp", "rh2m",
    "date_unix", "day_julian", "weekday", "hour",
]

# 2. Emission-related component

In [3]:
df_emi=nm.decom_emi(df_pre, value='value',model=model_flaml,feature_names=feature_names, n_samples=300)

In [4]:
df_emi

,observed,date_unix,day_julian,weekday,hour,emi_total,emi_noise,emi_base
date,,,,,,,,
2020-01-01 00:00:00,58.1,12.668870,3.871651,0.610155,0.517937,26.630335,-0.217863,9.179583
2020-01-01 01:00:00,43.2,12.993568,3.897367,0.616550,-0.277830,26.725071,0.315832,9.179583
2020-01-01 02:00:00,43.0,12.990789,3.919213,0.652529,-0.689368,26.269460,0.216714,9.179583
2020-01-01 03:00:00,42.8,13.605958,3.996255,0.756229,-0.544857,26.769277,-0.223892,9.179583
2020-01-01 04:00:00,36.8,12.023013,3.816854,0.614232,-0.482917,26.199798,1.049033,9.179583
...,...,...,...,...,...,...,...,...
2020-12-31 19:00:00,11.7,2.911798,1.918571,-0.037757,0.423198,13.451105,-0.944288,9.179583
2020-12-31 20:00:00,11.0,1.948353,1.806747,-0.024357,0.168251,13.280739,0.202161,9.179583
2020-12-31 21:00:00,15.3,2.017164,1.829870,-0.050080,-0.232918,13.242433,0.498814,9.179583


In [5]:
df_emi_lgb=nm.decom_emi(df_pre, value='value',model=model_lgb,feature_names=feature_names, n_samples=300)

In [6]:
df_emi_lgb

,observed,date_unix,day_julian,weekday,hour,emi_total,emi_noise,emi_base
date,,,,,,,,
2020-01-01 00:00:00,58.1,2.273492,0.837924,0.949954,-1.231277,11.782491,-0.188657,9.141057
2020-01-01 01:00:00,43.2,2.132183,0.871257,0.885486,-0.981591,12.351965,0.303574,9.141057
2020-01-01 02:00:00,43.0,1.633276,0.635153,0.897775,-1.373066,10.940541,0.006346,9.141057
2020-01-01 03:00:00,42.8,2.761802,0.823655,0.902410,-1.320456,12.254624,-0.053845,9.141057
2020-01-01 04:00:00,36.8,1.599306,0.791742,0.843972,-1.277735,12.153921,1.055580,9.141057
...,...,...,...,...,...,...,...,...
2020-12-31 19:00:00,11.7,1.243697,-0.126167,-0.122462,0.329669,9.893116,-0.572678,9.141057
2020-12-31 20:00:00,11.0,0.552977,-0.335630,-0.084750,0.443982,9.828062,0.110425,9.141057
2020-12-31 21:00:00,15.3,0.506469,-0.433505,-0.140636,0.031967,9.493860,0.388508,9.141057


In [7]:
df_emi_lgb_v2=nm.decom_emi(df_pre, value='value',backend='lightgbm',feature_names=feature_names, n_samples=300, model_config={"n_trials": 4, "cv_folds": 2, "nrounds": 80, "early_stopping_rounds": 10})

# 3. meteorological contributions

In [8]:
df_met=nm.decom_met(df_pre, value='value',model=model_flaml,feature_names=feature_names, n_samples=300)

In [9]:
df_met

,observed,emi_total,blh,u10,d2m,sp,v10,t2m,tcc,tp,rh2m,ssrd,met_total,met_base,met_noise
date,,,,,,,,,,,,,,,
2020-01-01 00:00:00,58.1,26.630335,1.281012,7.467686,0.194357,2.051034,8.698453,1.852040,3.124065,1.760500,2.528539,-28.957686,31.469665,-0.272253,31.741918
2020-01-01 01:00:00,43.2,26.725071,1.284851,6.231958,-0.381247,0.318169,5.602956,0.689621,2.240020,0.127640,0.021600,-16.135569,16.474929,-0.272253,16.747182
2020-01-01 02:00:00,43.0,26.269460,6.661535,4.877534,0.035659,0.292344,5.427349,-0.773047,3.159615,0.437907,1.331969,-21.450863,16.730540,-0.272253,17.002793
2020-01-01 03:00:00,42.8,26.769277,9.080918,1.461573,-0.536543,1.062392,2.821509,-1.973920,2.508594,0.436435,0.701793,-15.562751,16.030723,-0.272253,16.302976
2020-01-01 04:00:00,36.8,26.199798,6.845813,1.901762,-0.639423,-1.134654,1.834616,-1.269115,2.621993,0.436787,1.667533,-12.265310,10.600202,-0.272253,10.872455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-31 19:00:00,11.7,13.451105,-0.657509,-0.628142,-0.487892,0.395608,0.750942,-0.557842,-0.164043,0.316310,-0.512757,1.545326,-1.751105,-0.272253,-1.478852
2020-12-31 20:00:00,11.0,13.280739,-0.674500,-0.154806,-0.387796,0.259942,0.713287,-0.703536,-0.661056,0.101850,-0.580398,2.087012,-2.280739,-0.272253,-2.008486
2020-12-31 21:00:00,15.3,13.242433,-0.733437,-0.770918,-0.702856,0.283636,0.689874,-0.539069,-0.613176,0.121021,-0.371919,2.636844,2.057567,-0.272253,2.329820


In [10]:
df_met_lgb=nm.decom_met(df_pre, value='value',model=model_lgb,feature_names=feature_names, n_samples=300)

In [11]:
df_met_lgb

,observed,emi_total,blh,u10,sp,d2m,v10,t2m,rh2m,ssrd,tcc,tp,met_total,met_base,met_noise
date,,,,,,,,,,,,,,,
2020-01-01 00:00:00,58.1,11.782491,-0.685416,8.038762,1.640852,0.753514,3.825544,1.172199,2.525825,-1.945932,0.655985,-15.981332,46.317509,0.420586,45.896923
2020-01-01 01:00:00,43.2,12.351965,-0.329723,6.929910,2.215440,1.391189,3.555435,1.345201,2.456192,-1.462857,0.780501,-16.881288,30.848035,0.420586,30.427449
2020-01-01 02:00:00,43.0,10.940541,1.292858,7.240248,2.302985,1.343468,4.232495,0.709927,2.269443,-1.188303,-0.121690,-18.081432,32.059459,0.420586,31.638874
2020-01-01 03:00:00,42.8,12.254624,4.516371,5.852926,1.700393,1.368153,3.804799,1.097481,1.922999,-2.512657,-0.023009,-17.727456,30.545376,0.420586,30.124791
2020-01-01 04:00:00,36.8,12.153921,5.451666,5.703379,1.322747,1.313811,3.879354,0.615520,1.964988,-2.201667,-0.068050,-17.981747,24.646079,0.420586,24.225493
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-31 19:00:00,11.7,9.893116,-0.713614,0.077028,0.993757,-0.459651,1.472640,1.488174,0.605461,1.392785,0.159464,-5.016044,1.806884,0.420586,1.386298
2020-12-31 20:00:00,11.0,9.828062,-0.778074,-0.272550,1.018316,-0.617715,1.245255,1.530570,0.232367,1.463422,0.034867,-3.856457,1.171938,0.420586,0.751352
2020-12-31 21:00:00,15.3,9.493860,-0.702614,-1.030734,1.085878,-0.581865,1.689764,1.340266,-0.277333,1.395077,-0.205498,-2.712942,5.806140,0.420586,5.385554


In [12]:
df_met.describe()

,observed,emi_total,blh,u10,d2m,sp,v10,t2m,tcc,tp,rh2m,ssrd,met_total,met_base,met_noise
count,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6.373000e+03,6.373000e+03
mean,9.134238,9.406491,0.519416,-0.652701,-0.076963,0.222474,0.065419,0.019378,-0.044578,-0.013108,-0.012347,-0.026989,-0.272253,-2.722530e-01,-7.135530e-17
std,8.137577,3.010506,3.470751,2.533220,1.970903,1.192657,1.383263,1.145386,0.701611,1.031286,0.907072,6.923077,7.185232,5.551551e-17,7.185232e+00
min,-4.300000,4.368258,-4.382790,-7.244844,-4.596844,-3.733670,-8.994465,-8.784062,-5.008630,-6.105995,-4.215600,-54.821196,-16.527500,-2.722530e-01,-1.625525e+01
25%,4.400000,6.718301,-1.441132,-2.078854,-0.759369,-0.264107,-0.678925,-0.484970,-0.361898,-0.377196,-0.467010,-1.760791,-4.389570,-2.722530e-01,-4.117317e+00
50%,6.800000,9.132554,-0.817697,-0.542834,-0.219742,0.101699,0.211821,-0.062977,-0.050119,-0.057379,-0.027103,1.349671,-1.597049,-2.722530e-01,-1.324796e+00
75%,10.700000,11.642586,0.725572,0.263720,0.155118,0.503422,0.832028,0.378424,0.264393,0.285756,0.391067,3.879176,1.575044,-2.722530e-01,1.847297e+00
max,72.300000,27.647050,17.678039,13.283862,20.516086,11.210346,8.698453,6.098200,5.550178,9.114521,11.291565,13.781787,56.285616,-2.722530e-01,5.655787e+01


In [13]:
df_met_lgb.describe()

,observed,emi_total,blh,u10,sp,d2m,v10,t2m,rh2m,ssrd,tcc,tp,met_total,met_base,met_noise
count,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6373.000000,6.373000e+03
mean,9.134238,8.713652,0.208575,-0.292080,0.323704,0.102159,0.161832,-0.010219,0.101140,-0.128301,-0.008433,-0.458376,0.420586,0.420586,2.140659e-16
std,8.137577,2.772825,2.839536,2.371714,1.099698,1.978052,1.246067,1.005361,0.907752,0.854979,0.768555,6.100074,7.412907,0.000000,7.412907e+00
min,-4.300000,3.302347,-3.343816,-6.067150,-3.770161,-4.865498,-3.749028,-4.538755,-3.687918,-3.102752,-3.088106,-40.371693,-13.566287,0.420586,-1.398687e+01
25%,4.400000,6.160516,-1.446899,-1.880425,-0.293541,-0.703401,-0.665057,-0.565993,-0.419564,-0.625912,-0.426106,-2.468778,-3.948782,0.420586,-4.369368e+00
50%,6.800000,8.830754,-0.891355,-0.475463,0.302535,-0.069506,0.285855,-0.066696,0.170154,-0.089603,-0.034881,0.687257,-1.022718,0.420586,-1.443304e+00
75%,10.700000,10.497763,0.359707,0.460143,0.893380,0.489680,0.983921,0.491303,0.710908,0.418752,0.388694,3.405776,2.345181,0.420586,1.924595e+00
max,72.300000,17.979330,10.921401,11.391471,5.435586,9.179952,5.580631,3.910938,3.838977,2.913822,4.271856,12.713022,56.597742,0.420586,5.617716e+01


In [14]:
df_met_v2=nm.decom_met(df_pre, value='value',backend='flaml',feature_names=feature_names, n_samples=300, model_config={"time_budget": 10, "metric": "r2", "estimator_list": ["lgbm"], "task": "regression"})

# 4. Decompose contributions

In [15]:
df_emi_flaml=nm.decompose(method='emission',df=df_pre, value='value',model=model_flaml,feature_names=feature_names, n_samples=300)

In [16]:
df_met_decompose=nm.decompose(method='meteorology',df=df_pre, value='value',model=model_flaml,feature_names=feature_names, n_samples=300)